## Context Window and Memory

The context window is the total amount of text an LLM can see at once.  
LLMs are stateless — they have no memory between API calls by default.  
Memory management is your responsibility as the developer.

## How it actually works

**Context window:**  
Everything the model can see in one API call —  
system prompt + conversation history + retrieved documents + current question + response.  
All of it counts toward the limit.  
Claude Sonnet 4: 200,000 tokens (~150,000 words).

**Why LLMs are stateless:**  
Each API call is completely independent.  
The model has no memory of previous calls.  
Send a message, get a response — that's it.  
Next call starts completely fresh unless you provide history.

**How conversation memory works:**  
You maintain history yourself.  
Every API call includes all previous messages:  
User: "What is a posting profile?"  
Assistant: "A posting profile is..."  
User: "How do I configure one?"  ← model needs previous context to answer this  

Without history in the API call — the model doesn't know what "one" refers to.

**Three memory strategies:**

**1. Full history:**  
Send every message from the beginning every time.  
Simple. Works for short conversations.  
Expensive and hits context limit for long conversations.

**2. Sliding window:**  
Keep only the last N messages.  
Drop the oldest when limit is reached.  
Loses early context but stays within token limits.

**3. Summarisation:**  
Periodically summarise older conversation into a short paragraph.  
Replace raw history with the summary.  
Preserves key context without using full tokens.

## Where it breaks or gets dangerous

**Context window overflow:**  
Too much text injected — system prompt + history + RAG chunks + question  
exceeds the context limit.  
API returns an error or silently truncates your input.  
Model gives wrong answer because it never saw part of your prompt.

**Lost in the middle:**  
Attention weakens for content in the middle of very long contexts.  
Model pays more attention to beginning and end.  
Critical instructions buried in the middle may be ignored.  
Always put the most important instructions at the start or end of your prompt.

**Memory confusion:**  
Without proper history management, the model loses track of conversation context.  
"What about the previous invoice?" — model has no idea what "previous" means.  
Retrieves wrong information. Gives irrelevant answer.

**Token cost:**  
Every token in context costs money.  
Sending full conversation history on every call gets expensive fast.  
In production, token management is a cost engineering problem, not just a technical one.

**Privacy risk:**  
Conversation history sent to the API contains everything the user said.  
In enterprise systems — financial data, vendor names, transaction details.  
Must handle history with the same data governance as any sensitive system.

In [ ]:
# Context Window and Memory — illustration

def count_tokens(text):
    """Rough token estimate: ~0.75 words per token"""
    words = len(text.split())
    return int(words / 0.75)

def manage_conversation_history(messages, max_tokens=4000, strategy="sliding_window"):
    """
    Manages conversation history to stay within token limits.
    """
    total_tokens = sum(count_tokens(m["content"]) for m in messages)
    
    if strategy == "full_history":
        return messages, total_tokens
    
    elif strategy == "sliding_window":
        while total_tokens > max_tokens and len(messages) > 1:
            messages = messages[1:]  # drop oldest message
            total_tokens = sum(count_tokens(m["content"]) for m in messages)
        return messages, total_tokens
    
    elif strategy == "summarise":
        if total_tokens > max_tokens:
            # In production, you'd call the LLM to summarise
            summary = "Summary: User asked about D365 posting profiles and configuration steps."
            messages = [{"role": "system", "content": summary}] + messages[-2:]
            total_tokens = sum(count_tokens(m["content"]) for m in messages)
        return messages, total_tokens

# Simulated conversation history
conversation = [
    {"role": "user", "content": "What is a posting profile in D365?"},
    {"role": "assistant", "content": "A posting profile in D365 defines how transactions are posted to the general ledger. It maps transaction types to specific ledger accounts based on rules you configure."},
    {"role": "user", "content": "How do I configure one for vendor payments?"},
    {"role": "assistant", "content": "To configure a posting profile for vendor payments, go to Accounts Payable > Setup > Vendor posting profiles. Create a new profile, set the summary account for vendor balances, and configure the offset accounts for payment differences."},
    {"role": "user", "content": "What about the settlement account?"},
]

print("Context Window and Memory Management")
print("=" * 60)
print(f"Total messages: {len(conversation)}")
print(f"Total tokens (estimate): {sum(count_tokens(m['content']) for m in conversation)}")
print()

for strategy in ["full_history", "sliding_window", "summarise"]:
    managed, tokens = manage_conversation_history(
        conversation.copy(), max_tokens=100, strategy=strategy
    )
    print(f"Strategy: {strategy}")
    print(f"Messages kept: {len(managed)}, Tokens: {tokens}")
    print()

print("Key insight:")
print("Every API call must include conversation history explicitly.")
print("LLMs have no memory between calls — you manage state.")
print(f"Claude Sonnet 4 context window: 200,000 tokens")

## My D365 analogy

In D365, a session timeout logs you out.  
When you log back in, the system remembers nothing  
about what you were doing before.  
You start fresh — unless the application explicitly saved your state.

LLMs work exactly the same way.  
Every API call is a new session.  
No memory of what came before — unless you send it explicitly.

Managing conversation history in an AI application  
is like implementing session state management in a D365 customisation.  
You decide what to save, what to discard, and how long to keep it.

The difference: in D365 you save to a database.  
In an LLM application you send history back in every API call.  
Same concept, different mechanism.

## LinkedIn post idea

Here is something that surprises most people about AI assistants:  

The model remembers nothing between conversations.  
Every API call starts completely fresh.  
When Claude remembers what you said earlier in a chat —  
that's not the model's memory.  
That's the application sending your conversation history  
back to the model on every single call.

As a developer building on the Claude API,  
this is your responsibility to manage.  
Too little history — the model loses context and gives wrong answers.  
Too much history — you hit token limits and costs spiral.  

In enterprise D365 AI applications,  
conversation history also contains sensitive financial data.  
Vendor names. Transaction amounts. Approval decisions.  
History management is not just a technical problem.  
It is a data governance problem.  

The details that matter in enterprise AI  
are never the ones the demos show you.